# Modèle alternatif — ResNet convolutif dilaté (semaine → semaine)

Candidat face au GRU bidirectionnel de `timeseries_net`. **Mêmes données, même découpage, même vecteur statique, mêmes cibles** — seul le modèle change, pour que la comparaison soit honnête.

## Pourquoi

Deux mesures faites sur ce jeu :
- l'erreur est **majoritairement une erreur de forme** : avec un niveau parfait par bâtiment, `total` ne monterait que de 0.794 à 0.848 ;
- le GRU unidirectionnel plafonnait à l'époque 4 **quelle que soit sa taille** (64 et 128 unités : même val_loss). Le bidirectionnel débloque jusqu'à l'époque 17. Le facteur limitant est l'accès à l'information dans la séquence, pas la capacité.

[Predict the element instead of the sequence](https://www.sciencedirect.com/science/article/pii/S0306261926003910) (Applied Energy, 2026) montre sur le même type de problème qu'un **ResNet convolutif** bat nettement les approches séquence-à-séquence récurrentes.

## L'adaptation retenue

Le papier prédit **une heure** à partir d'une courte fenêtre. Ici on garde **la semaine entière en sortie** : c'est la plus petite unité contenant les deux régimes de consigne (`_get_masks` renvoie un masque `weekday` et un masque `weekend` distincts) et les deux régimes d'occupation.

On garde donc 168 h en entrée et en sortie, mais on remplace la récurrence par des **convolutions dilatées résiduelles** : chaque heure de sortie est calculée depuis un voisinage de ~253 h, en parallèle, sans goulot séquentiel.

In [ ]:
import copy, time
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA   = Path().resolve().parent.parent / 'data' / 'processed'
TS_BASE = ('https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/'
           'end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/'
           'timeseries_individual_buildings/by_state/upgrade=0')

SEQ_LEN = 168      # fenêtre = 1 semaine (contient les régimes semaine ET week-end)

# Pas des fenêtres d'ENTRAÎNEMENT. 168 = fenêtres disjointes, exactement comme timeseries_net :
# c'est la seule façon de comparer l'architecture toutes choses égales par ailleurs.
# Passer à 24 (~360 fenêtres/bâtiment) est un SECOND essai, à mener après, séparément.
STRIDE  = 168

N_OUT   = 4
# 64 canaux -> ~155 000 paramètres, soit le même budget que le GRU bidirectionnel (164 000).
# Mesuré sur ce CPU : 31 s/époque, contre 82 s à 128 canaux et 58 s pour le GRU bidirectionnel.
CANAUX  = 64
DILATIONS = (1, 2, 4, 8, 16, 32)
NOMS    = ['total', 'chauffage', 'clim', 'eau_chaude']

# champ réceptif : chaque bloc a 2 convs de noyau 3 et dilatation d -> +4d
RF = 1 + 4 * sum(DILATIONS)
print(f'champ réceptif théorique : {RF} h  (doit dépasser {SEQ_LEN})')

## 1. Le modèle

```
 f(t)  (B, 168, 31)  --transpose-->  (B, 31, 168)
 s     (B, 52)       --tile-------->  (B, 52, 168)
                                          |
                                    concat (83 canaux)
                                          |
                              Conv1d 83 -> 64, noyau 1
                                          |
              6 blocs résiduels, dilatations 1, 2, 4, 8, 16, 32
                                          |
                              Conv1d 64 -> 4, noyau 1  + Softplus
                                          |
                       (B, 4, 168) --transpose--> (B, 168, 4)
```

- `padding='same'` symétrique → **non causal** : chaque heure voit son passé *et* son futur, comme le GRU bidirectionnel.
- Le vecteur statique est injecté **dès l'entrée** (et non seulement dans la tête), pour que les convolutions puissent moduler leur traitement selon le bâtiment à toutes les profondeurs.
- Les dilatations 1→32 donnent un champ réceptif de **253 h**, soit plus que la semaine : chaque heure prédite voit toute sa fenêtre. Ça couvre les constantes de temps thermiques du parc, mesurées de 4 h à 65 h.

In [ ]:
class Bloc(nn.Module):
    """Bloc résiduel : 2 convolutions dilatées, non causales."""
    def __init__(self, ch, dilation):
        super().__init__()
        self.c1 = nn.Conv1d(ch, ch, 3, padding='same', dilation=dilation)
        self.b1 = nn.BatchNorm1d(ch)
        self.c2 = nn.Conv1d(ch, ch, 3, padding='same', dilation=dilation)
        self.b2 = nn.BatchNorm1d(ch)
        self.act = nn.GELU()

    def forward(self, x):
        h = self.act(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return self.act(x + h)              # connexion résiduelle


class LoadConv(nn.Module):
    """ResNet convolutif dilaté : f(t) + statique -> conso(t). Sortie >= 0."""
    def __init__(self, n_time, n_static, ch=CANAUX, n_out=N_OUT, dilations=DILATIONS):
        super().__init__()
        self.entree = nn.Conv1d(n_time + n_static, ch, 1)
        self.blocs  = nn.Sequential(*[Bloc(ch, d) for d in dilations])
        self.sortie = nn.Sequential(nn.Conv1d(ch, n_out, 1), nn.Softplus())

    def forward(self, x_time, static):
        x = x_time.transpose(1, 2)                                  # (B, n_time, T)
        s = static.unsqueeze(-1).expand(-1, -1, x.size(-1))         # (B, n_static, T)
        h = self.entree(torch.cat([x, s], dim=1))
        h = self.blocs(h)
        return self.sortie(h).transpose(1, 2)                       # (B, T, n_out)


# contrôle de forme sur des données factices
_m = LoadConv(31, 52)
_y = _m(torch.randn(2, SEQ_LEN, 31), torch.randn(2, 52))
print('sortie :', tuple(_y.shape), '| min =', float(_y.min()), '(doit être >= 0)')
print(f'paramètres : {sum(p.numel() for p in _m.parameters()):,}')

## 2. Données — identiques à `timeseries_net`

`load_building` est recopiée telle quelle. Si tu modifies `f(t)` dans `timeseries_net`, il faut la remettre à jour ici, sinon la comparaison n'a plus de sens.

In [ ]:
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
SETP = ['out.schedules.heating_setpoint..c', 'out.schedules.cooling_setpoint..c']
TGT  = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
        ['total', 'heating', 'cooling', 'hot_water']]

def load_building(bldg_id, state):
    """Recopié de timeseries_net (cellule c5)."""
    path = DATA / f'{bldg_id}-0.parquet'
    ts = pd.read_parquet(path if path.exists() else f'{TS_BASE}/state={state}/{bldg_id}-0.parquet')
    ts['timestamp'] = pd.to_datetime(ts['timestamp'])
    brut = ts.set_index('timestamp').reindex(columns=WEA + SCHED + SETP + TGT)
    h = brut[WEA + SCHED + SETP].resample('1h').mean().iloc[:8760]   # grandeurs instantanées
    y = brut[TGT].resample('1h').sum().iloc[:8760]                   # énergies -> somme
    h[SCHED] = h[SCHED].fillna(0.0)
    if h[SETP].isna().any().any():
        raise ValueError(f'bâtiment {bldg_id} : consignes absentes. Relancer '
                         'extraction_timeseries_oedi avec FORCE = True.')
    i = h.index
    cal = pd.DataFrame({
        'h_sin': np.sin(2*np.pi*i.hour/24),      'h_cos': np.cos(2*np.pi*i.hour/24),
        'd_sin': np.sin(2*np.pi*i.dayofweek/7),  'd_cos': np.cos(2*np.pi*i.dayofweek/7),
        'm_sin': np.sin(2*np.pi*(i.month-1)/12), 'm_cos': np.cos(2*np.pi*(i.month-1)/12),
    }, index=i)
    t_ext = h[WEA[0]]
    ecart = pd.DataFrame({
        'ecart_chauffage': (h[SETP[0]] - t_ext).clip(lower=0),
        'ecart_clim':      (t_ext - h[SETP[1]]).clip(lower=0),
    }, index=i)
    return pd.concat([h[WEA + SCHED + SETP], cal, ecart], axis=1), y

In [ ]:
# On garde les années COMPLÈTES par bâtiment (une seule copie en mémoire) et on découpe
# les fenêtres à la volée. Avec STRIDE = 24 et 503 bâtiments, matérialiser les ~180 000
# fenêtres coûterait ~3,8 Go ; ici on reste à ~550 Mo.
BUILDINGS = list(pd.read_csv(DATA / 'nn_buildings.csv').itertuples(index=False, name=None))

t0 = time.time()
F, YB, BIDS = [], [], []
for bid, st in BUILDINGS:
    f, y = load_building(bid, st)
    F.append(f.values.astype('float32'))
    YB.append(y.values.astype('float32'))
    BIDS.append(bid)
F, YB, BIDS = np.stack(F), np.stack(YB), np.array(BIDS)
COLONNES = list(f.columns)

print(f'F {F.shape} | Y {YB.shape} | {time.time()-t0:.0f}s | '
      f'{F.nbytes/1e6:.0f} Mo + {YB.nbytes/1e6:.0f} Mo')

In [ ]:
# vecteur statique : identique à timeseries_net (cellule c7)
preds = pd.read_parquet(DATA / 'static_preds.parquet').set_index('bldg_id')
feat  = pd.read_parquet(DATA / 'X_47features.parquet')
feat  = feat.assign(tau=feat['C'] / (feat['UA'] + feat['H_ve']) / 3.6)
assert set(BIDS) <= set(feat.index)

# --- Présence de l'équipement ------------------------------------------------------------
# 79/503 bâtiments (16 %) ont un chauffe-eau NON électrique, 26/503 (5 %) n'ont pas de clim :
# leur consommation de cet usage est nulle toute l'année. `in.water_heater_fuel` et
# `in.hvac_cooling_type` ne sont PAS dans les 48 features (retirés par DROP_DHW dans
# lgbm_electricity_5features) — le réseau n'avait aucun moyen de le savoir.
equip = pd.read_parquet(DATA / 'metadata_clean.parquet',
                        columns=['bldg_id', 'in.hvac_cooling_type', 'in.water_heater_fuel']
                        ).set_index('bldg_id')
a_clim = (equip.loc[BIDS, 'in.hvac_cooling_type'] != 'None').values.astype('float32')
ecs_el = (equip.loc[BIDS, 'in.water_heater_fuel'] == 'Electricity').values.astype('float32')

S = np.hstack([preds.loc[BIDS].values, feat.loc[BIDS].values,
               a_clim[:, None], ecs_el[:, None]]).astype('float32')        # (n_bat, 54)

# Masque des sorties [total, chauffage, clim, eau_chaude] : pas d'équipement -> zéro exact.
# Contrainte physique, pas apprise. Nécessaire car Softplus tend vers 0 sans l'atteindre.
un = np.ones_like(a_clim)
G = np.stack([un, un, a_clim, ecs_el], axis=-1).astype('float32')          # (n_bat, 4)

# split PAR BÂTIMENT — indispensable ici : les fenêtres se chevauchent, un split par fenêtre
# mettrait les mêmes heures des deux côtés. Même graine que timeseries_net.
rng   = np.random.default_rng(42)
val_b = set(rng.choice(BIDS, size=max(1, round(0.2 * len(BIDS))), replace=False))
est_val = np.array([b in val_b for b in BIDS])
i_tr, i_va = np.where(~est_val)[0], np.where(est_val)[0]

# standardisation ajustée sur les bâtiments d'ENTRAÎNEMENT uniquement
xm, xs = F[i_tr].mean((0, 1), keepdims=True), F[i_tr].std((0, 1), keepdims=True) + 1e-8
sm, ss = S[i_tr].mean(0, keepdims=True),      S[i_tr].std(0, keepdims=True) + 1e-8
ys     = YB[i_tr].std((0, 1), keepdims=True) + 1e-8
Fn, Sn, Yn = (F - xm) / xs, (S - sm) / ss, YB / ys      # cibles : pas de centrage (Softplus)

print(f'{len(BIDS)} bâtiments | {len(i_tr)} entraînement | {len(i_va)} validation')
print(f'f(t) = {F.shape[-1]} entrées | s = {S.shape[1]} colonnes | min(Yn) = {Yn.min():.3f}')
print(f'sans clim : {int((a_clim == 0).sum())} bâtiments | sans ECS électrique : {int((ecs_el == 0).sum())}')

In [ ]:
class Fenetres(Dataset):
    """Fenêtres d'une semaine découpées à la volée dans les années complètes."""
    def __init__(self, idx_bat, stride, L=SEQ_LEN):
        self.L = L
        self.pos = [(b, d) for b in idx_bat
                    for d in range(0, Fn.shape[1] - L + 1, stride)]

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, k):
        b, d = self.pos[k]
        sl = slice(d, d + self.L)
        return (torch.from_numpy(Fn[b][sl]), torch.from_numpy(Sn[b]),
                torch.from_numpy(Yn[b][sl]), torch.from_numpy(G[b]))


# entraînement : fenêtres glissantes (plus d'exemples)
# validation   : fenêtres DISJOINTES (stride = 168), exactement comme timeseries_net,
#                pour que les R² soient directement comparables
ds_tr = Fenetres(i_tr, STRIDE)
ds_va = Fenetres(i_va, SEQ_LEN)
print(f'entraînement : {len(ds_tr):,} fenêtres (stride {STRIDE})')
print(f'validation   : {len(ds_va):,} fenêtres (stride {SEQ_LEN}, disjointes)')

## 3. Vérification du champ réceptif

On perturbe **une seule heure** de l'entrée et on regarde sur combien d'heures de sortie ça se répercute. Doit dépasser 168 h, sinon les dilatations ne couvrent pas la semaine.

In [ ]:
_m = LoadConv(F.shape[-1], S.shape[1]).eval()
x = torch.zeros(1, SEQ_LEN, F.shape[-1], requires_grad=True)
s = torch.zeros(1, S.shape[1])
_m(x, s)[0, SEQ_LEN // 2].sum().backward()          # gradient de l'heure du milieu
touchees = (x.grad[0].abs().sum(-1) > 0).sum().item()
print(f"l'heure {SEQ_LEN//2} dépend de {touchees} heures d'entrée sur {SEQ_LEN}")
print(f'champ réceptif théorique : {RF} h -> la semaine entière est couverte'
      if touchees == SEQ_LEN else 'ATTENTION : champ réceptif insuffisant')

## 4. Entraînement

In [ ]:
torch.manual_seed(0)
ld_tr = DataLoader(ds_tr, batch_size=64, shuffle=True,
                   generator=torch.Generator().manual_seed(0))
ld_va = DataLoader(ds_va, batch_size=128)

# le masque (B, 4) s'applique à toutes les heures -> unsqueeze(1) diffuse sur les 168
masque = lambda p, g: p * g.unsqueeze(1)

model = LoadConv(F.shape[-1], S.shape[1]).to(device)
opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
print(f'{sum(p.numel() for p in model.parameters()):,} paramètres')

PATIENCE = 5
best, best_state, best_ep = float('inf'), None, -1
for epoch in range(40):
    model.train(); t0 = time.time()
    for xb, sb, yb, gb in ld_tr:
        xb, sb, yb, gb = xb.to(device), sb.to(device), yb.to(device), gb.to(device)
        opt.zero_grad(); lossf(masque(model(xb, sb), gb), yb).backward(); opt.step()
    model.eval(); tot = n = 0
    with torch.no_grad():
        for xb, sb, yb, gb in ld_va:
            p = masque(model(xb.to(device), sb.to(device)), gb.to(device))
            tot += lossf(p, yb.to(device)).item() * len(xb); n += len(xb)
    vloss = tot / n
    print(f'epoch {epoch:3d}  val_loss {vloss:.4f}  ({time.time()-t0:.0f}s)')

    if vloss < best:
        best, best_ep = vloss, epoch
        best_state = copy.deepcopy(model.state_dict())
    elif epoch - best_ep >= PATIENCE:
        print(f'arrêt anticipé (aucun progrès depuis {PATIENCE} époques)'); break

model.load_state_dict(best_state)
print(f'meilleure val_loss {best:.4f} (époque {best_ep})')

## 5. Comparaison au GRU bidirectionnel

Chiffres à battre, obtenus sur le **même découpage** avec `timeseries_net` :
`total 0.834 | chauffage 0.835 | clim 0.886 | eau_chaude 0.821`, val_loss `0.1711`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

model.eval(); P, T, GG = [], [], []
with torch.no_grad():
    for xb, sb, yb, gb in ld_va:
        P.append(masque(model(xb.to(device), sb.to(device)), gb.to(device)).cpu().numpy() * ys)
        T.append(yb.numpy() * ys); GG.append(gb.numpy())
pred, true, gate = np.concatenate(P), np.concatenate(T), np.concatenate(GG)

REF = {'total': 0.833, 'chauffage': 0.826, 'clim': 0.904, 'eau_chaude': 0.853}
print('=== Validation (bâtiments jamais vus) ===')
print(f'{"":12}{"avec masque":>13}{"sans masque":>13}{"écart":>9}')
for i, n in enumerate(NOMS):
    r = r2_score(true[..., i].ravel(), pred[..., i].ravel())
    print(f'{n:12}{r:13.3f}{REF[n]:13.3f}{r-REF[n]:+9.3f}')

# contrôle : les usages absents doivent être prédits exactement à zéro
for i, n in [(2, 'clim'), (3, 'eau_chaude')]:
    absent = gate[:, i] == 0
    if absent.any():
        print(f'  {n:11} {int(absent.sum())} fenêtres sans équipement -> '
              f'max prédit = {pred[absent, :, i].max():.1e} (doit être 0.0)')

w = 0
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for i, ax in enumerate(axes.ravel()):
    ax.plot(true[w, :, i], label='réel', lw=1.5)
    ax.plot(pred[w, :, i], label='prédit', lw=1.5, alpha=0.8)
    ax.set(title=NOMS[i], xlabel='heure', ylabel='kWh')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('ResNet convolutif — réel vs prédit, 1 semaine (validation)', fontsize=13)
plt.tight_layout(); plt.show()